In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# | default_exp process.semantics

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

InteractiveShell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
from pathlib import Path
import asyncio



In [ ]:
# | export
from ribosome.core.domclass import DOMClass

# process.semantics

>DOM semantics processing
output-file: process.semantics.html
title: Process Semantics

This notebook demonstrates a DOM semantics processing pipeline.
---

In [ ]:
# test walking the AST with line numbers
if False:
    md_file = "../res/siasun_md_sample/SN024002/SN024002《新松SN7B-7-0.90规格参数》A-1.md"
    print(f"Using file: {md_file}")
    md_file = Path(md_file)
    print(f"File exists: {md_file.exists()}")


    if md_file.exists():
        dom = DOMClass(md_file, ollama_client=ollama_client, openai_client=openai_client)
        dom.setup()
        print(f"Raw JSON length: {len(dom.raw_json) if dom.raw_json else 0}")
        print(f"AST JSON length: {len(dom.ast_json) if dom.ast_json else 0}")
        dom.walk_nodes_with_line_number()
        # jsoncfg.load_config(str(dom.ast_json_file))

In [ ]:
if False:
    md_file = Path(os.getcwd()).parent / r'res/md.json.semantics/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md'
    # md_file = Path(os.getcwd()).parent / r"res/md.mid.hrsl.test/06 产品手册/机械类/SR122001Installation and Maintenance Manual of Siasun SR12 Series Industrial Robot-A.0_20250304110301/SR122001Installation and Maintenance Manual of Siasun SR12 Series Industrial Robot-A.0_20250304110301.md" 
    # md_file = Path("/v/data/新型机器人智能问答系统数据源-md/.md.mid.hrsl.test.not.processed/DSS_MD_CoC_25_HEL_014 Issue 1/DSS_MD_CoC_25_HEL_014 Issue 1.md") 
    # md_file = Path('/d/devel/rag/ribosome/res/md.hrsl/01 设计标准/SX047001新松机器人产品识别设计标准A-1/SX047001新松机器人产品识别设计标准A-1.md') 
    
    # md_file = Path(os.getcwd()).parent / 'res/siasun_md_sample_hrsl/SR02400401/SR02400401《 SIASUN SR210A-210-2.65 Specifications-CE》A-0.md'
    # md_file = Path(os.getcwd()).parent / 'res/siasun_md_sample_hrsl/SX322002/SX322002.md'
    # md_file = Path(os.getcwd()).parent / 'res/siasun_md_sample_hrsl/SR024011/SR024011.md'
    # md_file = Path(os.getcwd()).parent / 'res/siasun_md_sample_hrsl/SN024002/SN024002.md'
    # md_file = Path(os.getcwd()).parent / 'res/siasun_md_sample/SN024002/SN024002.md'
    print(f"Using file: {md_file}")
    print(f"File exists: {md_file.exists()}")
    
    if md_file.exists():
        dom = DOMClass(md_file, ollama_client=ollama_client, openai_client=openai_client)
        dom.setup()
        print(f"Raw JSON length: {len(dom.raw_json) if dom.raw_json else 0}")
        print(f"AST JSON length: {len(dom.ast_json) if dom.ast_json else 0}")
        
        js_sections_file = md_file.parent / (str(md_file.stem) + "_ast.json")
        print(f"Saving to: {js_sections_file}")
        # Write the JSON representation of the AST to the file
        js_sections_file.write_text(dom.ast_json, encoding="utf-8")
        
        # Test textualize functionality
        print("Starting textualization...")
        await dom.textualize()
        
        js_semantics_file = md_file.parent / (str(md_file.stem) + "_semantics.json")
        js_semantics_file.write_text(dom.ast_json, encoding="utf-8")
        print(f"Semantics saved to: {js_semantics_file}")
    else:
        print("File does not exist!")

In [ ]:
def document_reorg(root_folder: Path | str) -> None:
    """Generate reorganized AST JSON files for every markdown file under a root folder.
    
    Args:
        root_folder: Root directory scanned recursively for markdown files.
    """
    root = Path(root_folder) if isinstance(root_folder, str) else root_folder
    for file in root.rglob("*.md"):
        dom = DOMClass(file, ollama_client=ollama_client, openai_client=openai_client)
        dom.setup()  # Load the Markdown content and convert it to JSON AST
        ast_json_file = file.parent / (str(file.stem) + "_ast.json")
        if not ast_json_file.exists() and dom.ast_json:
            ast_json_file.write_text(dom.ast_json, encoding="utf-8")
            print(f"Processing file: {file}")
        elif ast_json_file.exists():
            # print(f"File already processed: {file}")
            pass
        else:
            print(f"AST JSON missing: {file}")

# document_reorg(Path("/v/data/新型机器人智能问答系统数据源-md/.md.mid.hrsl.test.not.processed"))

# document_reorg(Path("../res/md.mid.hrsl.test"))
# document_reorg(Path("/v/data/documents-semantics/.md"))
# document_reorg(Path("../res/SN024002"))
# document_reorg(Path("../res/test_batch_async"))
# document_reorg(Path("/v/data/documents-semantics/.md.hrsl"))
# document_reorg(Path("../res/siasun_md_sample_hrsl"))

In [ ]:
#| export
async def analyze_one_document_async(md_file: Path, semaphore: asyncio.Semaphore) -> DOMClass:
    """Analyze one markdown document asynchronously and persist its semantics JSON.
    
    Args:
        md_file: Markdown file to analyze.
        semaphore: Concurrency guard used while processing the file.
    
    Returns:
        DOMClass: Processed DOMClass object with analysis status populated.
    """
    try:
        async with semaphore:  # Limit concurrent access to the semaphore
            print(f"Analyzing document: {md_file}")
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                dom = DOMClass(md_file, ollama_client=ollama_client, openai_client=openai_client)
                dom.setup()  # Load the Markdown content and convert it to JSON AST
                if w:
                    print(f"Warnings encountered while setting up DOMClass for {md_file}: {w}")
            if dom.semantics_json:
                dom.analysis_status.status = ResponseStatus.COMPLETED
                dom.analysis_status.exception = f"Document already analyzed: {md_file.stem}"
                return dom

            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                await dom.textualize()  # Summarize the document
                if w:
                    print(f"Warnings encountered while textualizing DOMClass for {md_file}: {w}")
            if dom.file_path:
                semantics_json_file = md_file.parent / (str(md_file.stem) + "_semantics.json")
                semantics_json_file.write_text(dom.ast_json, encoding="utf-8")  # type: ignore
                dom.analysis_status.status = ResponseStatus.COMPLETED  # type: ignore
                dom.analysis_status.exception = f"Finished analyzing document: {md_file.stem}"
    except asyncio.CancelledError:
        dom.analysis_status.status = ResponseStatus.CANCELLED  # type: ignore
        dom.analysis_status.exception = f"Analysis cancelled for document: {md_file}"  # type: ignore
    except Exception as e:
        dom.analysis_status.status = ResponseStatus.ERROR  # type: ignore
        dom.analysis_status.exception = f"Error occurred while analyzing document: {md_file}, Error: {e}"  # type: ignore

    return dom  # type: ignore

In [ ]:
from ollama import embed


def check_one_document(md_file: Path, embed: bool = False) -> DOMClass | None:
    """Inspect a markdown document and decide whether it still needs analysis or embedding.
    
    Args:
        md_file: Markdown file to inspect.
        embed: When `True`, check for embedding readiness instead of semantic analysis.
    
    Returns:
        DOMClass | None: DOMClass object when further work is needed, otherwise `None`.
    """
    print(f"Checking document: {md_file}")
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        dom = DOMClass(md_file, ollama_client=ollama_client, openai_client=openai_client)
        dom.setup()  # Load the Markdown content and convert it to JSON AST
        if w:
            print(f"Warnings encountered while setting up DOMClass for {md_file}: {w}")

    if embed:
        return dom if dom.semantics_json else None
    else:
        return None if dom.semantics_json else dom


In [ ]:
import tqdm

async def document_semantics_analysis(root_folder: Path) -> None:
    """Run semantic analysis for all pending markdown files under a root folder.
    
    Args:
        root_folder: Root directory scanned recursively for markdown files.
    """
    
    semaphore = asyncio.Semaphore(1)  # Limit the number of concurrent tasks
    # Iterate through all Markdown files in the root folder recursively
    # and create a task for each file to process it asynchronously
    # files = list(root_folder.rglob("*.md"))
    # to_do = [analyze_one_document_async(file, semaphore) for file in files]

    doms_to_analyze = [check_one_document(file, embed=False) for file in root_folder.rglob("*.md")]
    to_do = [analyze_one_document_async(dom.file_path, semaphore) for dom in doms_to_analyze if dom is not None]  # type: ignore
    to_do_iter = asyncio.as_completed(to_do)  # Create an iterator for the tasks
    to_do_iter = tqdm.tqdm(to_do_iter, total=len(to_do), desc="Processing files", unit="file")
    for coro in to_do_iter:
        # Wait for each task to complete and get the result
        try:
            dom = await coro  # Await the completion of the task
        except Exception as e:
            # print(f"Error processing file: {dom.file_path}")
            print(f"Error processing file: {e}")
            continue
        print(f"Processed file: {dom.file_path} with title: {dom.title}, "
            f"status: {dom.analysis_status.status}, exception: {dom.analysis_status.exception}")

# Run the document semantics analysis
# await document_semantics_analysis(Path("../res/md.json.semantics/06 产品手册"))
await document_semantics_analysis(Path("../res/md.json.semantics"))

In [ ]:
#| export
async def embed_one_document_async(md_file: Path, semaphore: asyncio.Semaphore) -> DOMClass:
    """Embed one markdown document asynchronously from its semantics JSON.
    
    Args:
        md_file: Markdown file whose semantics should be embedded.
        semaphore: Concurrency guard used while processing the file.
    
    Returns:
        DOMClass: Processed DOMClass object with embedding status populated.
    """
    try:
        async with semaphore:  # Limit concurrent access to the semaphore
            print(f"Analyzing document: {md_file}")
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                dom = DOMClass(md_file, ollama_client=ollama_client, openai_client=openai_client)
                dom.setup()  # Load the Markdown content and convert it to JSON AST
                if w:
                    print(f"Warnings encountered while setting up DOMClass for {md_file}: {w}")
            if not dom.semantics_json:
                dom.analysis_status.status = ResponseStatus.PENDING
                dom.analysis_status.exception = f"Document semantics not found: {md_file.stem}"
                return dom

            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                await dom.embed()  # Summarize the document
                if w:
                    print(f"Warnings encountered while embedding DOMClass for {md_file}: {w}")
            dom.analysis_status.status = ResponseStatus.COMPLETED  # type: ignore
            dom.analysis_status.exception = f"Finished embedding document: {md_file.stem}"
    except asyncio.CancelledError:
        dom.analysis_status.status = ResponseStatus.CANCELLED  # type: ignore
        dom.analysis_status.exception = f"Embedding cancelled for document: {md_file}"  # type: ignore
    except Exception as e:
        dom.analysis_status.status = ResponseStatus.ERROR  # type: ignore
        dom.analysis_status.exception = f"Error occurred while embedding document: {md_file}, Error: {e}"  # type: ignore

    return dom  # type: ignore

In [ ]:
import tqdm

async def document_embedding(root_folder: Path) -> None:
    """Run embedding generation for all eligible markdown files under a root folder.
    
    Args:
        root_folder: Root directory scanned recursively for markdown files.
    """
    
    semaphore = asyncio.Semaphore(2)  # Limit the number of concurrent tasks
    # Iterate through all Markdown files in the root folder recursively
    # and create a task for each file to process it asynchronously
    # files = list(root_folder.rglob("*.md"))
    # to_do = [analyze_one_document_async(file, semaphore) for file in files]

    doms_to_embed = [check_one_document(file, embed=True) for file in root_folder.rglob("*.md")]
    to_do = [embed_one_document_async(dom.file_path, semaphore) for dom in doms_to_embed if dom is not None]  # type: ignore
    to_do_iter = asyncio.as_completed(to_do)  # Create an iterator for the tasks
    to_do_iter = tqdm.tqdm(to_do_iter, total=len(to_do), desc="Processing files", unit="file")
    for coro in to_do_iter:
        # Wait for each task to complete and get the result
        try:
            dom = await coro  # Await the completion of the task
        except Exception as e:
            # print(f"Error processing file: {dom.file_path}")
            print(f"Error processing file: {e}")
            continue
        print(f"Processed file: {dom.file_path} with title: {dom.title}, "
              f"status: {dom.analysis_status.status}, exception: {dom.analysis_status.exception}")

# Run the document embedding
await document_embedding(Path("../res/md.json.semantics"))

## asyncio interface of processing many markdown files


In [ ]:
async def process_one(file: Path) -> None:
    """Process one markdown file end to end and write its AST and semantics artifacts.
    
    Args:
        file: Markdown file to process.
    """
    print(f"Processing file started: {file}")
    dom = DOMClass(file, ollama_client=ollama_client, openai_client=openai_client)
    dom.setup()
    ast_json_file = file.parent / (str(file.stem) + "_ast.json")
    semantics_json_file = file.parent / (str(file.stem) + "_semantics.json")
    ast_json_file.write_text(dom.ast_json, encoding="utf-8")
    await dom.textualize()
    semantics_json_file.write_text(dom.ast_json, encoding="utf-8")
    print(f"Semantics analysis completed for {file}. Results saved to {semantics_json_file}")

In [ ]:
async def supervisor(root_folder: Path) -> int:
    """Launch processing tasks for all markdown files under a root folder.
    
    Args:
        root_folder: Root directory scanned recursively for markdown files.
    
    Returns:
        int: Number of completed processing tasks.
    """

    tasks = []
    for file in root_folder.rglob("*.md"):
        print(f"Processing file started: {file}")
        tasks.append(process_one(file))
    
    res = await asyncio.gather(*tasks)

    return len(res)

In [ ]:
def process_many(root_folder: Path) -> None:
    """Run the markdown processing supervisor from synchronous code.
    
    Args:
        root_folder: Root directory scanned recursively for markdown files.
    """
    
    return asyncio.run(supervisor(root_folder))

# process_many(Path("../res/test_batch"))

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()